In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import joblib

In [2]:
df = pd.read_csv("dataset.csv")

df.head()

,session_id,network_packet_size,protocol_type,login_attempts,session_duration,encryption_used,ip_reputation_score,failed_logins,browser_type,unusual_time_access,attack_detected
0,SID_00001,599,TCP,4,492.983263,DES,0.606818,1,Edge,0,1
1,SID_00002,472,TCP,3,1557.996461,DES,0.301569,0,Firefox,0,0
2,SID_00003,629,TCP,3,75.044262,DES,0.739164,2,Chrome,0,1
3,SID_00004,804,UDP,4,601.248835,DES,0.123267,0,Unknown,0,1
4,SID_00005,453,TCP,5,532.540888,AES,0.054874,1,Firefox,0,0


In [3]:
df["failed_login_count"] = df["failed_logins"]

df["login_success_ratio"] = (
    (df["login_attempts"] - df["failed_logins"])
    / (df["login_attempts"] + 1)
)

df["privilege_target"] = (
    df["ip_reputation_score"] < 0.3
).astype(int)

df["hour_of_day"] = np.random.randint(0,24,len(df))

df["is_off_hours"] = df["unusual_time_access"]

df["action_frequency"] = 1 / (df["session_duration"] + 1)

df["connection_rate"] = (
    df["network_packet_size"]
    / (df["session_duration"] + 1)
)

df["bytes_transferred"] = df["network_packet_size"]

df["is_new_ip"] = (
    df["ip_reputation_score"] < 0.2
).astype(int)

In [4]:
features = [

"failed_login_count",
"login_success_ratio",
"privilege_target",
"hour_of_day",
"is_off_hours",
"action_frequency",
"connection_rate",
"bytes_transferred",
"is_new_ip"

]

X = df[features]

y = df["attack_detected"]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
model = RandomForestClassifier(

n_estimators = 400,
max_depth = 12,
class_weight="balanced",
random_state = 60

)

model.fit(

X_train,
y_train

)

RandomForestClassifier(class_weight='balanced', max_depth=12, n_estimators=400,
                       random_state=60)

In [13]:
predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

print(classification_report(y_test, predictions))

Accuracy: 0.8118448637316562
              precision    recall  f1-score   support

           0       0.75      0.98      0.85      1042
           1       0.96      0.61      0.75       866

    accuracy                           0.81      1908
   macro avg       0.86      0.79      0.80      1908
weighted avg       0.85      0.81      0.80      1908



In [34]:
joblib.dump(
    model,
    "../rf_model.joblib"
)

print("rf_model.joblib saved")

rf_model.joblib saved


In [ ]:
sample_df = pd.DataFrame({
"failed_login_count":[1],
"login_success_ratio":[0.55],
"privilege_target":[0],
"hour_of_day":[13],
"is_off_hours":[0],
"action_frequency":[0.002],
"connection_rate":[1],
"bytes_transferred":[520],
"is_new_ip":[0]

})

prob = model.predict_proba(sample_df)

risk_score = prob[0][1] * 100

# print(X.describe())
print("Risk Score:", risk_score)

       failed_login_count  login_success_ratio  privilege_target  hour_of_day  \
count         9537.000000          9537.000000       9537.000000  9537.000000   
mean             1.517773             0.405726          0.471008    11.432211   
std              1.033988             0.381509          0.499185     6.896925   
min              0.000000            -2.000000          0.000000     0.000000   
25%              1.000000             0.250000          0.000000     5.000000   
50%              1.000000             0.500000          0.000000    11.000000   
75%              2.000000             0.666667          1.000000    17.000000   
max              5.000000             0.928571          1.000000    23.000000   

       is_off_hours  action_frequency  connection_rate  bytes_transferred  \
count   9537.000000       9537.000000      9537.000000        9537.000000   
mean       0.149942          0.007396         3.711136         500.430639   
std        0.357034          0.032358  

In [5]:
df["attack_detected"].value_counts(normalize=True)
X.describe()

NameError: name 'X' is not defined